# 4 Smart Charging Using Reinforcement Learning

The previous sections studied *where* and *when* ride-hailing demand occurs. This section turns to the operational side of an electric fleet: given that a vehicle must be ready for its shift, *how should it charge*? We leave the Chicago trip data behind and study a single electric taxi that charges at home. The driver arrives at 2 p.m. and leaves at 4 p.m., so there is a two-hour window in which a charging agent sets the charging power every 15 minutes, giving eight sequential decisions. The energy the vehicle will need for the coming day is uncertain and is revealed only at departure, drawn from a normal distribution. Charging cost grows exponentially with power, and running out of energy is heavily penalised.

This makes the problem a sequential decision under uncertainty, which we formalise as a Markov decision process and solve with reinforcement learning. The agent has to balance two opposing forces: charge enough to cover an uncertain demand (safety), while spreading the load and exploiting cheaper time slots to keep cost down (economy).

The questions guiding this section are:

1. How is home charging formalised as a Markov decision process (states, actions, reward)?
2. Which charging policy minimises cost while avoiding energy shortfall under uncertain demand?
3. How close to the provable optimum does a learned DQN policy get, and does it beat naive charging strategies?
4. How does the policy react to the electricity price structure and to the level of demand uncertainty?

The section is organised as follows:

- *4.1 Problem Formalisation (MDP)*
- *4.2 Environment Implementation*
- *4.3 Design Choices and Parameters*
- *4.4 Reinforcement Learning Solution (DQN)*
- *4.5 Results: Policy and Evaluation against Baselines*
- *4.6 Sensitivity and Discussion*

The charging demand is modelled synthetically and independently of the trip data; where useful we anchor its parameters to realistic daily driving so the setup stays grounded rather than arbitrary.

## 4.1 Problem Formalisation (MDP)

Before any code, the charging problem is written out as a complete Markov decision process. This formalisation is the backbone of the section: the environment in 4.2 and the agent in 4.4 must both implement exactly this definition. The following has to be specified:

- **State** `s_t = (t, SoC_t)`: the current 15-minute slot `t` in {0, ..., 7} and the battery state of charge `SoC_t` in kWh (discretised). The slot belongs in the state because the electricity price varies over time, so the best action depends on *when* we are.
- **Action** `a_t`: the charging power for the slot, taken from a small discrete set (zero, low, medium, high in kW) so that value-based methods such as DQN apply directly.
- **Transition:** charging is deterministic, `SoC_{t+1} = min(B, SoC_t + a_t * 0.25)` over a 15-minute slot, capped at battery capacity `B`. The only stochastic element is the daily energy demand `D ~ N(mu, sigma)`, drawn once at departure (after slot 7).
- **Reward:** the per-slot cost `-alpha_t * exp(a_t)` following the assignment's exponential cost, plus a terminal penalty `-P` applied when `SoC_final < D`, i.e. the vehicle cannot cover its day.
- **Horizon and discount:** a finite horizon of eight steps with discount `gamma = 1`, since every decision within one short session matters equally and there is no long future to discount.

State the Markov property explicitly (the state carries everything needed to decide, so the history is irrelevant) and list every assumption (fixed window, charging only, no driving in between, demand revealed only at the end). Present the MDP as one compact table so a reader grasps the whole decision problem at a glance.

## 4.2 Environment Implementation

Implement the MDP from 4.1 as a small, self-contained simulation with a clean interface, so the *same* environment can be driven by the RL agent and by every baseline. This is what makes the later comparison fair. The cell below has to deliver:

- A class exposing `reset()` and `step(action)` in the style of a Gym environment. `step` applies the action, updates the state of charge, accumulates the slot cost, advances the clock, and on the final step draws the demand `D` and adds the shortfall penalty if needed. It returns `(next_state, reward, done, info)`.
- A seedable random generator for the demand draw, so every run is reproducible.
- An `info` dictionary logging per-step cost, state of charge, the drawn demand and whether a shortfall occurred, so 4.5 can analyse trajectories without re-running the training.
- A sanity check: run one fixed dummy policy (for example always *medium*), print the resulting trajectory and final cost, and confirm that the dynamics, the cap at `B` and the penalty all behave exactly as specified in 4.1.

Keep the environment deliberately simple and free of any agent logic. All intelligence belongs in 4.4.

In [6]:
import numpy as np

class ChargingEnv:
    # Charging power levels (kW) for each discrete action index
    SLOT_HOURS    = 0.25    # 15 minutes = 0.25 hours

    def __init__(self, params: dict, seed: int = 42):
        """
        Parameters
        ----------
        params : dict with keys:
            mu, sigma     – demand distribution (kWh)
            B             – battery capacity (kWh)
            soc_init      – initial SoC at 14:00 (kWh)
            P             – shortfall penalty
            alpha         – list of 8 price coefficients (one per slot)
            action_powers – list of charging power levels for each action (kW). For example, [0, 3, 11, 22] for off/low/medium/high.
            n_actions     – number of discrete actions (length of action_powers)
        seed : int
            Random seed for reproducible demand draws.
        """
        self.mu       = params['mu']
        self.sigma    = params['sigma']
        self.B        = params['B']
        self.soc_init = params['soc_init']
        self.P        = params['P']
        self.alpha    = np.array(params['alpha'])
        self.n_slots  = len(self.alpha)
        self.action_powers = params['action_powers']
        self.n_actions     = len(self.action_powers)

        # Seedable RNG — same seed → same demand sequence across runs
        self.rng = np.random.default_rng(seed)

        self.reset()

    # reset: start a new charging session
    def reset(self):
        """Reset to 14:00 with initial SoC. Returns the starting state."""
        self.soc = self.soc_init
        self.t = 0
        self.total_cost = 0.0
        self.trajectory = [] # empty diary, new day starts
        return self._state()

    def step(self, action: int):
        """
        Apply action for the current slot.

        Parameters
        ----------
        action : int  (0=off, 1=low, 2=medium, 3=high)

        Returns
        -------
        next_state : tuple (t+1, soc_{t+1})
        reward     : float  (negative cost; includes terminal penalty if done)
        done       : bool
        info       : dict   (diagnostic log for further analysis)
        """
        assert 0 <= action < self.n_actions, f"Invalid action {action}"

        power = self.action_powers[action]   # kW

        # 1. Update battery
        energy_added = power * self.SLOT_HOURS        # kWh
        prev_soc     = self.soc
        self.soc     = min(self.B, self.soc + energy_added)

        # 2. Electricity cost — exponential in action index (not raw kW)
        slot_cost = self.alpha[self.t] * (np.exp(action) - 1)
        reward    = -slot_cost

        # 3. Advance clock
        prev_t  = self.t
        self.t += 1
        self.total_cost += slot_cost

        # 4. Terminal step: draw demand and check for shortfall
        done      = (self.t == self.n_slots)
        demand    = None
        shortfall = False
        if done:
            demand    = float(self.rng.normal(self.mu, self.sigma))
            shortfall = (self.soc < demand)
            if shortfall:
                reward -= self.P

        # 5. Log this step
        info = {
            'slot'       : prev_t,
            'time_label' : f"{14 + prev_t // 4}:{(prev_t % 4) * 15:02d}",
            'action'     : action,
            'power_kw'   : power,
            'soc_before' : prev_soc,
            'soc_after'  : self.soc,
            'slot_cost'  : slot_cost,
            'demand'     : demand,
            'shortfall'  : shortfall,
        }
        self.trajectory.append(info)

        return self._state(), reward, done, info

    def _state(self):
        """Return current state as (t, SoC) with SoC rounded to 1 decimal."""
        return (self.t, round(self.soc, 1))

    def render_trajectory(self):
        """Print a human-readable episode summary."""
        header = f"{'Time':<8}{'Action':<8}{'kW':<6}{'SoC before':>12}{'SoC after':>11}{'Slot cost':>11}"
        print(header)
        print('-' * len(header))
        for s in self.trajectory:
            print(f"{s['time_label']:<8}{s['action']:<8}{s['power_kw']:<6}"
                  f"{s['soc_before']:>12.2f}{s['soc_after']:>11.2f}{s['slot_cost']:>11.4f}")
        last = self.trajectory[-1]
        print('-' * len(header))
        print(f"Final SoC : {last['soc_after']:.2f} kWh")
        print(f"Demand    : {last['demand']:.2f} kWh")
        print(f"Shortfall : {last['shortfall']}")
        print(f"Total cost: {self.total_cost:.4f}")

print("ChargingEnv class defined.")


ChargingEnv class defined.


### Sanity check

Before any learning, we run one **dummy policy**
and print the full trajectory. This confirms:
- The battery increments correctly each slot
- The battery cap `B` works (SoC never exceeds B)
- The slot cost uses `α_t × exp(action)` — higher at peak slots
- The shortfall penalty fires when `SoC_final < D`

In [ ]:

_params_preview = {
    'mu'      : 30.0,
    'sigma'   : 5.0,
    'B'       : 25.0,
    'soc_init': 5.0,
    'P'       : 500.0,
    'alpha'   : [0.05, 0.05, 0.08, 0.12, 0.18, 0.18, 0.12, 0.08],
    'action_powers': [0, 3, 11, 22],
}

env_check = ChargingEnv(_params_preview, seed=0)
env_check.reset()

done = False
total_reward = 0.0
while not done:
    _, reward, done, _ = env_check.step(action=2)   
    total_reward += reward

env_check.render_trajectory()


Time    Action  kW      SoC before  SoC after  Slot cost
--------------------------------------------------------
14:00   2       11            5.00       7.75     0.3195
14:15   2       11            7.75      10.50     0.3195
14:30   2       11           10.50      13.25     0.5111
14:45   2       11           13.25      16.00     0.7667
15:00   2       11           16.00      18.75     1.1500
15:15   2       11           18.75      21.50     1.1500
15:30   2       11           21.50      24.25     0.7667
15:45   2       11           24.25      25.00     0.5111
--------------------------------------------------------
Final SoC : 25.00 kWh
Demand    : 30.63 kWh
Shortfall : True
Total cost: 5.4946


## 4.3 Design Choices and Parameters

Fix and *justify* every parameter of the environment. This is where the problem is made non-trivial, so the choices are argued, not merely stated. This subsection has to provide:

- A parameter table: demand mean `mu` and standard deviation `sigma`, battery capacity `B`, maximum power and the discrete action levels in kW, the initial state of charge at 2 p.m., the shortfall penalty `P`, and the state-of-charge discretisation step.

- A feasibility argument: with a 22 kW cap over two hours at most 44 kWh can be added, so `B` and the initial charge are chosen such that covering `mu` plus a safety buffer is actually reachable inside the window.
- A calibration of the penalty `P` so that it strictly dominates the largest plausible charging cost, which guarantees the agent never trades safety for a few cents of savings.


**4.3.1. Demand distribution**

**4.3.2. Battery capacity**
The battery must be large enough to cover demand on most days, with a safety buffer

B=40 kWh covers mu+2sigma=40 kWh, ensuring 95% of demand days are satisfiable when fully charged. This is consistent with compact urban EV battery sizes (e.g. Renault Zoe: 52 kWh, VW ID.3: 45 kWh)."

**4.3.3. Initial SoC at 2PM** It is the remaining battery after the morning shift. The driver has been working all morning. The battery is nearly depleted by the time they get home.
SoC_init = 5 kWh  (nearly empty — about 12.5% of B=40)

4.3.4. Action levels - charging powers
We set:
Action 0:  0 kW  — charger off
Action 1:  3 kW  — standard household socket (slow charging)
Action 2: 11 kW  — home wallbox (typical residential charger)
Action 3: 22 kW  — fast home charger (upper residential limit)


**4.3.2. Electricity price**

To have reasonable and realistic electricity price profile, we get real each 5-minute price data from Commonwealth Edison (ComEd) API. ComEd provides electricity delivery services to more than 3.8 million residential and business customers across Northern Illinois, including Chicago.

By using real pricing data from ComEd, the model can have actual information of electricity market in Chicago.

In [ ]:
import requests
import pandas as pd
import numpy as np

# 1. Fetch each 5-minute price data from ComEd API from 2024/01/01 to 2026/04/30 (unit: cents/kWh)
url = "https://hourlypricing.comed.com/api?type=5minutefeed&datestart=202401010001&dateend=202604302359"

response = requests.get(url, timeout=60)
data = response.json()
print(f"Got {len(data)} records")

# 2. Parse into DataFrame, convert millis to timestamp
df = pd.DataFrame(data)
print("Overview of first 5 records:")
print(df.head(5))

df["timestamp"]    = pd.to_datetime(df["millisUTC"].astype(int), unit="ms", utc=True).dt.tz_convert("America/Chicago")
df["price_cents"]  = df["price"].astype(float)
df["hour"]         = df["timestamp"].dt.hour
df["minute"]       = df["timestamp"].dt.minute

print(df[["timestamp", "price_cents"]].head())

# 3. Filter to 14:00–15:59 window and assign slots
df_window = df[df["hour"].between(14, 15)].copy()
df_window["slot"] = ((df_window["hour"] - 14) * 60 + df_window["minute"]) // 15
df_window = df_window[df_window["slot"].between(0, 7)]

# 4. Compute alpha_t: average price per slot
slot_avg  = df_window.groupby("slot")["price_cents"].mean()
alpha     = (slot_avg / 100).round(4).values   # convert from cents to dollars

# 5. Print results
labels = ['14:00','14:15','14:30','14:45','15:00','15:15','15:30','15:45']
print("\nAlpha profile:")
for label, a in zip(labels, alpha):
    bar = '█' * int(a * 200)
    print(f"  {label}  alpha={a:.4f}  {bar}")

# 6. Save data
df.to_csv("../data/comed_prices_raw.csv", index=False)
pd.DataFrame({"slot": range(8), "time": labels, "alpha_t": alpha}).to_csv("../data/comed_alpha_profile.csv", index=False)
print("\nSaved: ../data/comed_prices_raw.csv, ../data/comed_alpha_profile.csv")

Got 243716 records
Overview of first 5 records:
       millisUTC price
0  1777611300000   3.8
1  1777611000000   3.4
2  1777610700000   3.6
3  1777610400000   2.9
4  1777610100000   3.5
                  timestamp  price_cents
0 2026-04-30 23:55:00-05:00          3.8
1 2026-04-30 23:50:00-05:00          3.4
2 2026-04-30 23:45:00-05:00          3.6
3 2026-04-30 23:40:00-05:00          2.9
4 2026-04-30 23:35:00-05:00          3.5

Alpha profile:
  14:00  alpha=0.0278  █████
  14:15  alpha=0.0309  ██████
  14:30  alpha=0.0324  ██████
  14:45  alpha=0.0328  ██████
  15:00  alpha=0.0301  ██████
  15:15  alpha=0.0342  ██████
  15:30  alpha=0.0372  ███████
  15:45  alpha=0.0403  ████████

Saved: ../data/comed_prices_raw.csv, ../data/comed_alpha_profile.csv


In [16]:
# Set parameters for the ChargingEnv
ALPHA = alpha.tolist()
TIMES = ['14:00','14:15','14:30','14:45','15:00','15:15','15:30','15:45']

In [ ]:
# Bundle into a single params dict (used by ChargingEnv and all agents) 
PARAMS = {
    'mu'      : MU,
    'sigma'   : SIGMA,
    'B'       : B,
    'soc_init': SOC_INIT,
    'P'       : P,
    'alpha'   : ALPHA,
}

## 4.4 Reinforcement Learning Solution (DQN)

Solve the MDP with a Deep Q-Network, the method the course covers for discrete-action control. This subsection has to deliver:

- A Q-network mapping the state to one Q-value per action, trained with epsilon-greedy exploration, an experience replay buffer and a target network. Report the architecture and all hyperparameters.
- Training over many episodes, each a fresh charging session with a newly drawn demand, plus a convergence plot (episode reward and loss) that demonstrates the agent actually learns rather than merely runs.
- A provable reference: because the state space is small, also compute the exact optimal policy by dynamic programming (value iteration over the discretised MDP). This optimum is the yardstick the DQN is measured against in 4.5 and is what lifts the evaluation from descriptive to rigorous.

Show the DQN learning curve approaching the dynamic-programming optimum, so convergence is quantified and not just asserted. A tabular Q-learning agent may be added as a lightweight second learner, but the DP optimum is the reference that matters.

## 4.5 Results: Policy and Evaluation against Baselines

Demonstrate that the learned policy is both safe and economical, and benchmark it properly. This subsection has to provide:

- A visualisation of the learned policy: the chosen action as a function of the state `(t, SoC)`, and the charging schedule of a representative episode, so the behaviour is readable rather than a black box.
- An evaluation over many independent test episodes on two metrics that capture the trade-off: mean recharging cost and shortfall rate (how often the vehicle runs out of energy). Reporting only one of the two is not enough.
- A benchmark table comparing the DQN against the dynamic-programming optimum and against naive baselines: constant charging, greedy charge-as-fast-as-possible, and a cheapest-slots heuristic. The DQN should sit close to the optimum and clearly dominate the naive strategies.
- Distributions, not only averages: show the spread of cost and the rare shortfall events, since a fleet operator cares about the worst cases, not just the mean.

This benchmarking layer is what turns "we built an RL agent" into "our agent is provably good", and it mirrors the evaluation rigour applied to the predictive models in Section 3.

## 4.6 Sensitivity and Discussion

Show that the result is robust and translate it into advice, with a few targeted experiments rather than an exhaustive grid. This subsection has to cover:

- Vary the electricity price profile from flat to steeply peaked and show how charging concentrates into the cheap slots as the profile steepens.
- Vary the demand uncertainty `sigma` and show that the policy keeps a larger safety buffer as uncertainty grows, making the safety-versus-cost trade-off explicit.
- Briefly vary the penalty `P` and the maximum power to confirm the policy still reacts sensibly.
- Interpret in plain terms what the agent has learned, then connect it to the business question of Section 5: when home charging is sufficient, and what this implies for the choice between private and public charging infrastructure.

Close by stating the limitations honestly (a single vehicle, a synthetic demand, a stylised price profile), so the scope of the conclusion is clear and not overstated.